# 05 — Build Model Inputs

**Thesis:** *Nowcasting and Indicator Selection in a Data-Rich Environment: An Application to German GDP Growth*

Writes the matrices that Part II actually uses from this notebook: `en_only` (DFM-EN) and `pls_only` (DFM-PLS). Also writes a ≥3/4 vote matrix (`core`) used to tune XGBoost. Publication lags are enforced in notebook 06, not by dropping series here.

The thesis does not identify a unique indicator set. The DFM comparison is EN vs block-balanced vs PLS vs frozen ifoCAST. Block-balanced and ifoCAST are built in the DFM scripts, not in this notebook.

| Matrix | Rule | Role in the thesis |
|---|---|---|
| `en_only` | Capped elastic net | DFM-EN |
| `pls_only` | PLS+VIP top 30 | DFM-PLS |
| `core` | ≥3 of EN, EN-smoothed, PLS, fixed-*k* | XGBoost hyperparameter tuning only |

A matrix-level intra-quarter lag gate would drop PLS’s lag-2 selections at M1/M2 and drive Jaccard overlap to zero. That is an artefact, not a finding.


## 1. Setup

`MIN_VOTES = 3`. Forecast window 2011M1–2025M12. Coverage threshold 30%, as in notebook 03.

Inputs: EN, EN-smoothed, PLS and **fixed-*k* = 30** matrices, plus metadata, the transformed panel and `pub_lag_map.csv`. The fourth voter is the Bai & Ng path, not the block-balanced *k* = 20 set.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Portable repository setup ---
_repo = next(
    (base for base in (Path.cwd(), *Path.cwd().parents)
     if (base / "src" / "german_gdp_nowcasting").is_dir()),
    None,
)
if _repo is None:
    raise RuntimeError("Could not locate src/german_gdp_nowcasting.")
_src = _repo / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from german_gdp_nowcasting.config import paths as _tp
from german_gdp_nowcasting.selection.core_utils import (
    build_coverage_mask,
    load_monthly_panel,
    load_pub_lag_map,
    make_monthly_forecast_origins,
)
from german_gdp_nowcasting.selection.dfm_input_builder import build_dfm_input_sets

MIN_VOTES = 3
FORECAST_START = '2011-01'
FORECAST_END = '2025-12'
MIN_COVERAGE = 0.30
_tp.SELECTION_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo    : {_tp.REPO_ROOT}')
print(f'Dataset : {_tp.DATASET_XLSX}')
print(f'Data    : {_tp.DATA}')
print(f'Outputs : {_tp.OUTPUTS}')


## 2. Load inputs

Loads the four binary matrices from notebooks 03–04 (same shape, same origin index, same column order), the enriched dictionary, the transformed panel and `pub_lag_map.csv`.

The lag map is passed through for API compatibility. `build_dfm_input_sets` does not drop series by publication lag. The ragged edge is applied in notebook 06.

PLS and fixed-*k* have 30 series at every origin by construction. EN size is data-driven; on the thesis specification it is 12–60 (mean 51.5). A maximum above 60 means the loaded CSV predates the cap.


In [ ]:
en_raw      = pd.read_csv(_tp.SELECTION_MATRIX_CSV,        index_col='forecast_origin').astype(int)
en_smoothed = pd.read_csv(_tp.EN_SMOOTHED_MATRIX_CSV,      index_col='forecast_origin').astype(int)
pls_matrix  = pd.read_csv(_tp.PLS_MATRIX_CSV,              index_col='forecast_origin').astype(int)
fixedk      = pd.read_csv(_tp.FIXEDK_MATRIX_CSV,           index_col='forecast_origin').astype(int)
matrices = {
    'EN raw':      en_raw,
    'EN smoothed': en_smoothed,
    'PLS':         pls_matrix,
    'fixed-k (k=30)':     fixedk,
}
for label, m in matrices.items():
    print(f'{label:12s}: {m.shape[0]} origins x {m.shape[1]} series  |  mean selected/origin = {m.sum(axis=1).mean():.1f}')
meta = pd.read_csv(_tp.DATA_DICT_ENRICHED_CSV, usecols=['id', 'name', 'category']).set_index('id')
X_monthly = load_monthly_panel(_tp.PANEL_TRANSFORMED_CSV)
pub_lag_map = load_pub_lag_map(_tp.PUB_LAG_CSV)
forecast_origins = make_monthly_forecast_origins(FORECAST_START, FORECAST_END)
coverage_mask = build_coverage_mask(X_monthly, forecast_origins, min_coverage=MIN_COVERAGE)
print(f'\nCoverage mask : {coverage_mask.shape}')
print(f'Pub-lag map   : {len(pub_lag_map)} series  (lag range {int(pub_lag_map.min())}-{int(pub_lag_map.max())} months)')


## 3. Build the matrices

`build_core_matrix` counts, at each origin, how many of the four loaded methods selected series *j*, and keeps *j* if that count is at least 3. That vote is an XGBoost convenience, not a claim that the methods recover one German core (Part I rank correlations 0.28–0.43).

`en_only` and `pls_only` are the corresponding raw matrices. No publication-lag filter is applied.


In [ ]:
sets = build_dfm_input_sets(
    matrices=matrices,
    meta=meta,
    coverage_mask=coverage_mask,
    pub_lag_map=pub_lag_map,
    min_votes=MIN_VOTES,
    en_label='EN raw',
    pls_label='PLS',
)
rate_table_diag = sets.pop('_rate_table_diagnostic')
# The diagnostic rate table below is full-sample and shown for inspection only.
print('Full-sample rate table (diagnostic, not used for core construction):')
print(rate_table_diag[['mean_across_methods', 'category']].sort_values('mean_across_methods', ascending=False).head(20).to_string())
print('\nSelected count per origin (mean / min / max):')
for k, m in sets.items():
    s = m.sum(axis=1)
    print(f'  {k:9s}: mean={s.mean():5.1f}  min={int(s.min()):3d}  max={int(s.max()):3d}')
if sets['en_only'].sum(axis=1).max() > 60:
    print('WARNING: en_only exceeds the thesis cap of 60. Cite 12–60 (mean 51.5), not this printout.')


## 4. Save

Writes `core_selection_matrix.csv`, `en_only_selection_matrix.csv` and `pls_only_selection_matrix.csv` under `outputs/indicator_selection/dfm_input_sets/`.

Notebook 06 loads `en_only` as DFM-EN. `pls_only` is the sensitivity set. `core` is not a DFM headline.


In [4]:
paths_out = {
    'core':     _tp.CORE_MATRIX_CSV,
    'en_only':  _tp.EN_ONLY_MATRIX_CSV,
    'pls_only': _tp.PLS_ONLY_MATRIX_CSV,
}
for key, path in paths_out.items():
    sets[key].to_csv(path)
    try:
        shown = path.relative_to(_tp.REPO_ROOT)
    except ValueError:
        shown = path
    print(f'  {key:9s} -> {shown}')


  core      -> outputs/indicator_selection/dfm_input_sets/core_selection_matrix.csv
  en_only   -> outputs/indicator_selection/dfm_input_sets/en_only_selection_matrix.csv
  pls_only  -> outputs/indicator_selection/dfm_input_sets/pls_only_selection_matrix.csv


## 5. Diagnostics

On the thesis specification, EN set size is 12–60 (mean 51.5). PLS is 30 at every origin. A loaded EN matrix with a maximum above 60 predates the cap.

Origin-by-origin Jaccard among `core`, `en_only` and `pls_only` is a description of these three files. It is not the thesis agreement statistic (Spearman of full-window weights, 0.28–0.43). High overlap between `core` and PLS is partly mechanical: PLS is one of the four voters.

The PC variance share at the last origin is a sanity check that the vote set has common-factor structure. The DFM uses two factors on `en_only`, not on `core`.


In [ ]:
import matplotlib.dates as mdates
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
_x = pd.to_datetime(sets['core'].index)
colors = {'core': '#1d4ed8', 'en_only': '#ea580c', 'pls_only': '#7c3aed'}
for key, m in sets.items():
    ax.plot(_x, m.sum(axis=1).values, label=key, lw=1.5, color=colors[key])
ax.set_xlabel('Forecast origin (month)')
ax.set_ylabel('# selected indicators')
ax.set_title('Selected count per origin — DFM input sets')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
ax = axes[1]
keys = list(sets.keys())
J = np.zeros((len(keys), len(keys)))
for i, ki in enumerate(keys):
    for j, kj in enumerate(keys):
        a = sets[ki].astype(bool)
        b = sets[kj].reindex(index=a.index, columns=a.columns, fill_value=0).astype(bool)
        inter = (a & b).sum(axis=1)
        union = (a | b).sum(axis=1).replace(0, np.nan)
        J[i, j] = (inter / union).median()
im = ax.imshow(J, vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys, rotation=25, ha='right')
ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys)
ax.set_title('Median Jaccard across origins')
for i in range(len(keys)):
    for j in range(len(keys)):
        ax.text(j, i, f'{J[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if J[i, j] > 0.6 else 'black')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
plt.tight_layout(); plt.show()

In [ ]:
# PC variance share for the core set at the last origin
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
last_origin = sets['core'].index[-1]
sel = sets['core'].columns[sets['core'].loc[last_origin].astype(bool)]
X_sel = X_monthly.loc[:, sel.intersection(X_monthly.columns)].copy()
X_sel = X_sel.loc[X_sel.notna().any(axis=1)]
X_arr = StandardScaler().fit_transform(SimpleImputer(strategy='mean').fit_transform(X_sel.values))
pca = PCA(n_components=min(5, X_arr.shape[1])).fit(X_arr)
var_share = pca.explained_variance_ratio_.cumsum()
print(f'Core set at {last_origin}: N = {X_sel.shape[1]} series, T = {X_sel.shape[0]} months')
for k, v in enumerate(var_share, start=1):
    print(f'  cumulative variance of first {k} PCs: {v:.3f}')

## 6. Next

`06_dfm_nowcasting.ipynb` estimates the mixed-frequency DFM on `en_only`. Block-balanced, ifoCAST, the equal-weight combination, DFM-TVP, DFM-SV, MLP-Factor and the AR variants are produced by the scripts listed at the end of that notebook.

Holding the DFM fixed, full-sample accuracy differences among EN, block-balanced, PLS and ifoCAST do not reject equal accuracy. After 2022 the average M1-to-M3 RMSFE profile inverts for every reported DFM.
